# Wav2Lip Lip-Sync on Colab **T4**

Makes a face (video or image) speak an audio clip. Perfect for making a
Wan-animated character talk with your Floyd / Cuffem / Player audio.

### This runs on the Colab T4 GPU only — NOT HuggingFace ZeroGPU.
It clones the open-source Wav2Lip code and runs `inference.py` on the T4
that Colab assigns to *your* runtime. It never calls the hosted HF Space,
so ZeroGPU (HF's shared, quota'd A100 slices) is never involved.

**First:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then run the
cells top to bottom. Step 0 proves you actually got a T4.

### Wide shot? You keep the WHOLE frame.
Wav2Lip only replaces the mouth and pastes it back onto the *full* original
frame — the output is your entire shot (e.g. the character walking across the
parking lot), just with the mouth moving. Nothing is cropped out. If a small,
distant face won't sync, use **Step 3.5 UPSCALE** (enlarges the whole frame,
keeps the full view) and, if needed, **BOX** in Step 4. Do **NOT** use CROP
for a wide shot — CROP is the only knob that discards the rest of the frame.


## Step 0 - Prove the GPU is a T4 (not ZeroGPU, not CPU)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.'
name = torch.cuda.get_device_name(0)
print('Torch is using:', name)
print('VERIFIED: running on', name, '- this is Colab hardware, not HF ZeroGPU.')


## Step 1 - Get Wav2Lip (maintained fork) and install deps
Uses `justinjohn0306/Wav2Lip`, which fixes the old-dependency problems and
hosts the model checkpoints.


In [ ]:
import os
if not os.path.isdir('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
%cd /content/Wav2Lip
!pip install -q -r requirements.txt
!pip install -q batch-face gdown
print('deps installed')


## Step 2 - Download the model checkpoints into the runtime


In [ ]:
%cd /content/Wav2Lip
import os
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('face_detection/detection/sfd', exist_ok=True)
B='https://github.com/justinjohn0306/Wav2Lip/releases/download/models/'
!wget -q -c $B'wav2lip.pth'     -O checkpoints/wav2lip.pth
!wget -q -c $B'wav2lip_gan.pth' -O checkpoints/wav2lip_gan.pth
!wget -q -c $B's3fd.pth'        -O face_detection/detection/sfd/s3fd.pth
!wget -q -c $B'mobilenet.pth'   -O checkpoints/mobilenet.pth
!wget -q -c 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth' -O checkpoints/GFPGANv1.4.pth
print('checkpoints ready:', os.listdir('checkpoints'))


## Step 3 - Upload your inputs
**FACE** = your clip or image (e.g. the wide Wan walking clip from
`D:\\MatrixVideos`). **AUDIO** = the voice line (wav/mp3, e.g. floyddeath.wav).


In [ ]:
from google.colab import files
print('Upload the FACE (mp4 / png / jpg):')
FACE = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('Upload the AUDIO (wav / mp3):')
AUDIO = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('FACE :', FACE)
print('AUDIO:', AUDIO)


## Step 3.5 - Upscale the input so a small/distant face is detectable
For a **wide shot**, this is the key step: it enlarges the *entire* frame,
so the full parking lot stays in view and the face just gains pixels. Set
`UPSCALE = 2` or `3` for a distant face; Step 5.5 shrinks the result back to
your target width. `UPSCALE = 1` skips it (fine for close-ups).


In [ ]:
UPSCALE = 3   # 1 = off. Use 2-3 for a small/distant face in a wide shot.
ORIG_W = None
if UPSCALE > 1:
    import os, subprocess
    # remember original width so we can shrink the result back later
    try:
        ORIG_W = int(subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0',
            '-show_entries','stream=width','-of','csv=p=0', FACE]).decode().strip().split(',')[0])
    except Exception:
        ORIG_W = None
    ext = os.path.splitext(FACE)[1].lower()
    up  = '/content/Wav2Lip/upscaled' + ext
    os.system(f'ffmpeg -y -loglevel error -i "{FACE}" -vf "scale=iw*{UPSCALE}:ih*{UPSCALE}:flags=lanczos" "{up}"')
    FACE = up
    print('upscaled', UPSCALE, 'x ->', FACE, '| original width:', ORIG_W)
else:
    print('upscale skipped (UPSCALE=1)')


## Step 4 - Run Wav2Lip on the T4
Focus knobs:
- **BOX** `[top,bottom,left,right]` (px, on the *upscaled* frame): force the
  face location, skip detection. Keeps the full frame. Use if UPSCALE alone
  still won't detect the face.
- **PADS** `[top,bottom,left,right]`: grow the box; raise bottom if the chin
  is clipped.
- **CROP** `[top,bottom,left,right]`: CROPS the output to that box. Only for a
  deliberate close-up — this DISCARDS the rest of the frame, so leave it
  `None` for a wide shot.

`wav2lip_gan.pth` = best mouth quality; `--nosmooth` helps single faces.


In [ ]:
BOX  = None            # e.g. [810, 1050, 1560, 1800] on the upscaled frame
PADS = [0, 15, 0, 0]   # top bottom left right
CROP = None            # leave None for a wide shot (CROP discards the rest)

cmd = ['python','inference.py',
       '--checkpoint_path','checkpoints/wav2lip_gan.pth',
       '--face', FACE, '--audio', AUDIO,
       '--outfile','/content/result.mp4',
       '--nosmooth','--resize_factor','1',
       '--pads', *map(str, PADS)]
if BOX:  cmd += ['--box',  *map(str, BOX)]
if CROP: cmd += ['--crop', *map(str, CROP)]
print('running:', ' '.join(cmd))
import subprocess; subprocess.run(cmd, check=True)
print('done -> /content/result.mp4')


## Step 5.5 - (wide shots) shrink the result back to your target width
If you upscaled in Step 3.5, bring the full-frame result back down to the
original size. Skips automatically if you didn't upscale.


In [ ]:
import os
FINAL = '/content/result.mp4'
if ORIG_W:
    FINAL = '/content/result_final.mp4'
    os.system(f'ffmpeg -y -loglevel error -i /content/result.mp4 -vf "scale={ORIG_W}:-2:flags=lanczos" -c:a copy "{FINAL}"')
    print('shrunk back to width', ORIG_W, '->', FINAL)
else:
    print('no downscale needed ->', FINAL)


## Step 6 - Preview and download the result


In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open(FINAL,'rb').read()).decode()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


In [ ]:
from google.colab import files
files.download(FINAL)  # saves to your Downloads; move it to D:\\MatrixVideos


---
### If the face still won't sync
1. Raise `UPSCALE` to 3-4 (Step 3.5).
2. Set `BOX` (Step 4) to the face rectangle *on the upscaled frame* — open
   the upscaled clip, read off the pixel box around the head.
3. `--out_height 720` on Step 4 adds GFPGAN sharpening (slower).

CROP is only for making a deliberate close-up; it removes the rest of the
frame, so don't use it when you want the whole parking-lot shot.
